# Day 10 - Tour from layouts to a complete GEMM

Official [Notebook 12](../notebooks/12_tour_to_sol_gemm.ipynb)
builds a Blackwell speed-of-light GEMM with TMA, UMMA, tensor
memory, software pipelining, and a vectorized epilogue. This
notebook teaches the same algorithmic roles with an
H100-compatible, single-stage FP32 SIMT GEMM.

The objective is to connect the earlier lessons:

```text
layouts -> CTA tiles -> cooperative copy -> shared memory
        -> per-thread MMA fragments -> K reduction -> epilogue
```

The local kernel favors visibility over peak throughput. The final
sections map each stage to the more specialized Blackwell version.

In [ ]:
from typing import Tuple

import torch

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

cutlass.cuda.initialize_cuda_context()

## 1. GEMM equation and the tensor convention

This lesson computes:

\[
C_{m,n} = \sum_{k=0}^{K-1} A_{m,k} B_{n,k}.
\]

Therefore A has logical shape `(M, K)`, B has logical shape
`(N, K)`, and C has shape `(M, N)`. B is represented as `(N, K)`
so the expression is `A @ B.T`.

Each output element performs K multiplies and K accumulations,
conventionally counted as `2*K` floating-point operations. Total
work is:

```text
FLOPs = 2 * M * N * K
```

In [ ]:
def gemm_work(m, n, k, element_bytes=4):
    flops = 2 * m * n * k
    minimum_bytes = (m * k + n * k + m * n) * element_bytes
    return flops, minimum_bytes, flops / minimum_bytes


for dims in ((256, 256, 64), (1024, 1024, 256), (8192, 8192, 8192)):
    flops, bytes_moved, intensity = gemm_work(*dims)
    print(
        f"{dims}: {flops / 1e9:.2f} GFLOP, "
        f"minimum traffic={bytes_moved / 1e6:.2f} MB, "
        f"intensity={intensity:.1f} FLOP/byte"
    )

## 2. The blocked-GEMM hierarchy

![Blocked GEMM](../notebooks/images/blocked_gemm.svg)

One CTA owns a `(BLK_M, BLK_N)` output tile and traverses K in
`BLK_K` chunks:

```text
grid tile (block_m, block_n)
    -> A tile: BLK_M x BLK_K
    -> B tile: BLK_N x BLK_K
    -> C tile: BLK_M x BLK_N
    -> repeat across K tiles
```

This lesson uses `(64, 64, 8)`. M and N must be multiples of 64 and
K must be a multiple of 8 because the challenge intentionally omits
edge predication. Day 04 showed how an identity coordinate tensor
would be used to remove that restriction.

## 3. Why the host tensors look transposed

PyTorch allocates row-major tensors, but this kernel wants stride 1
in the M mode for A/C and the N mode for B. It creates row-major
`(K, M)` storage and then permutes the view:

```python
a = torch.empty(K, M).permute(1, 0)  # logical (M,K), stride (1,M)
b = torch.empty(K, N).permute(1, 0)  # logical (N,K), stride (1,N)
c = torch.empty(N, M).permute(1, 0)  # logical (M,N), stride (1,M)
```

`permute` changes the view, not the stored values. Contiguous major
modes make each thread's four-Float32, 128-bit copy contiguous.

In [ ]:
M, N, K = 256, 256, 64
a_host = torch.empty(K, M).permute(1, 0)
b_host = torch.empty(K, N).permute(1, 0)
c_host = torch.empty(N, M).permute(1, 0)

print("A shape/stride:", a_host.shape, a_host.stride())
print("B shape/stride:", b_host.shape, b_host.stride())
print("C shape/stride:", c_host.shape, c_host.stride())
assert a_host.stride()[0] == 1
assert b_host.stride()[0] == 1
assert c_host.stride()[0] == 1

## 4. `local_tile` selects one CTA's views

A conceptual `(M,N,K)` CTA tiler must be projected onto tensors that
do not contain all three modes:

| Tensor | Logical modes | Projection | Result |
| --- | --- | --- | --- |
| A | `(M,K)` | `(1,None,1)` | `(BLK_M,BLK_K,k_tiles)` |
| B | `(N,K)` | `(None,1,1)` | `(BLK_N,BLK_K,k_tiles)` |
| C | `(M,N)` | `(1,1,None)` | `(BLK_M,BLK_N)` |

With `coord=(bidx,bidy,None)`, the CTA fixes its M/N grid location
while preserving the complete K-tile mode. The trailing K-tile
mode is the mainloop iteration space.

In [ ]:
def tile_counts(m, n, k, tile=(64, 64, 8)):
    return tuple(dim // block for dim, block in zip((m, n, k), tile))


counts = tile_counts(256, 256, 64)
assert counts == (4, 4, 8)
print(
    f"grid={counts[0]} x {counts[1]} CTAs; "
    f"each CTA performs {counts[2]} K-tile iterations"
)

## 5. Global-to-shared movement

A and B tiles are cooperatively copied into shared memory. A
`CopyAtom` describes one hardware copy operation; `TiledCopy`
distributes those operations across `(thread,value)` coordinates.

This kernel uses:

```python
cute.nvgpu.cpasync.CopyG2SOp()
num_bits_per_copy = 4 * 32  # four Float32 values = 128 bits
```

`thr_copy.partition_S(gA)` and `partition_D(sA)` create aligned
per-thread source and destination views. The views describe
ownership; `cute.copy` performs the transfer.

After issuing copies:

1. `cp_async_commit_group()` closes the copy group.
2. `cp_async_wait_group(0)` waits until no group remains pending.
3. `barrier()` makes the shared tile safe for all consumer threads.

## 6. Shared-to-register movement and MMA ownership

`TiledMma` describes how MMA work is distributed. Each thread first
obtains a slice:

```python
thr_mma = tiled_mma.get_slice(tidx)
```

Then:

- `partition_A(sA)` maps the shared A tile into that thread's MMA A view;
- `partition_B(sB)` does the same for B;
- `partition_C(gC)` maps accumulator ownership to output addresses;
- `make_fragment_A/B/C` allocates matching register fragments.

Partitioned tensors are mappings, not copies. Data enters the A/B
register fragments only when `autovec_copy` executes. The C
fragment must be initialized to zero before the K reduction.

## 7. The single-stage K mainloop

For every K tile:

```text
global A/B --cp.async--> shared A/B
        wait + CTA barrier
shared A/B --autovec_copy--> register A/B
register A/B --cute.gemm--> register accumulator
        CTA barrier before shared-memory overwrite
```

This ordering is intentionally conservative. The wait prevents
compute from reading incomplete data; the final barrier prevents
the next iteration from overwriting shared memory while another
thread still consumes it.

The inner `k_block` loop is compile-time unrolled because fragment
shapes are static. The outer `k_tile` loop depends on the problem's
dynamic K extent.

In [ ]:
BLK_M, BLK_N, BLK_K = 64, 64, 8
outputs_per_cta = BLK_M * BLK_N
fmas_per_k_tile = BLK_M * BLK_N * BLK_K
print(f"outputs per CTA: {outputs_per_cta}")
print(f"FMAs per K tile: {fmas_per_k_tile}")
print(f"FLOPs per K tile: {2 * fmas_per_k_tile}")

## 8. Epilogue: accumulator to global C

After all K tiles, `tCrC` contains the complete per-thread
accumulator. The simple epilogue copies it directly to the matching
`tCgC` global view.

Production epilogues often add:

- conversion from FP32 accumulation to FP16/BF16 output;
- scaling, bias, activation, or residual fusion;
- vectorized stores with explicit alignment and divisibility;
- accumulator subtiling to reduce register pressure.

Official Notebook 12 demonstrates why vector-store eligibility can
materially affect throughput. CuTe needs correct layout,
alignment, and divisibility information to select wide stores.

## 9. H100 lesson versus Blackwell SOL GEMM

| Algorithmic role | This notebook | Official Notebook 12 |
| --- | --- | --- |
| Global-to-shared load | Cooperative `cp.async` | TMA |
| Matrix operation | FP32 `MmaUniversalOp` | tcgen05/UMMA |
| Accumulator storage | Registers | Tensor memory |
| A/B synchronization | CTA barriers | TMA-UMMA pipelines |
| Staging | One stage | Multiple prefetched stages |
| Epilogue | Direct register-to-global copy | Subtiled/vectorized |

The Blackwell notebook is SM100-only. Do not try to execute its
TMA/UMMA/tensor-memory kernel on H100. The local challenge preserves
the decomposition while using mechanisms supported by the current
pod.

## 10. Challenge: connect the complete dataflow

Fill three connected regions:

1. **CTA tiles:** use `local_tile` with the correct projections.
2. **Ownership:** allocate shared tensors, partition copies/MMA, and
   allocate register fragments.
3. **Mainloop and epilogue:** move each K tile through memory spaces,
   accumulate it, and store C.

Track every variable by writing a sentence of the form:

```text
tAgA maps this thread's copy coordinates
to addresses in the CTA's global A tile.
```

If you cannot state both the domain and destination, inspect that
tensor before continuing.

In [ ]:
class SimpleSGemm:
    def __init__(self, cta_tiler: Tuple[int, int, int] = (64, 64, 8),
                 num_threads: int = 128):
        self.cta_tiler = cta_tiler
        self.bM, self.bN, self.bK = cta_tiler
        self.num_threads = num_threads

    @cute.jit
    def __call__(self, mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor):
        # smem layouts: M-major / N-major so MMA can vectorize smem->rmem loads.
        sA_layout = cute.make_layout((self.bM, self.bK), stride=(1, self.bM))
        sB_layout = cute.make_layout((self.bN, self.bK), stride=(1, self.bN))

        # gmem -> smem: 128b vector cp.async on M (for A) and N (for B).
        num_vec: cutlass.Constexpr = 4
        def make_g2s(tiler_lead, dtype):
            t = cute.make_layout((tiler_lead, self.num_threads // tiler_lead),
                                  stride=(1, tiler_lead))
            v = cute.make_layout((num_vec, 1))
            atom = cute.make_copy_atom(cute.nvgpu.cpasync.CopyG2SOp(),
                                        dtype,
                                        num_bits_per_copy=dtype.width * num_vec)
            return cute.make_tiled_copy_tv(atom, t, v)

        tiled_copy_A = make_g2s(self.bM // num_vec, mA.element_type)
        tiled_copy_B = make_g2s(self.bN // num_vec, mB.element_type)

        # Tiled MMA: 1x1x1 universal FMA atoms laid out as (T/16, 16, 1).
        op = cute.nvgpu.MmaUniversalOp(cutlass.Float32)
        atoms_layout = cute.make_layout((self.num_threads // 16, 16, 1),
                                         stride=(16, 1, 0))
        permutation = (cute.make_layout((atoms_layout.shape[0], 4), stride=(4, 1)),
                       cute.make_layout((atoms_layout.shape[1], 4), stride=(4, 1)),
                       None)
        tiled_mma = cute.make_tiled_mma(op, atoms_layout, permutation_mnk=permutation)

        grid = (*cute.ceil_div(mC.shape, (self.bM, self.bN)), 1)
        self.kernel(mA, mB, mC, sA_layout, sB_layout,
                    tiled_copy_A, tiled_copy_B, tiled_mma).launch(
            grid=grid, block=(cute.size(atoms_layout), 1, 1),
        )

    @cute.kernel
    def kernel(self, mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor,
               sA_layout: cute.Layout, sB_layout: cute.Layout,
               tiled_copy_A: cute.TiledCopy, tiled_copy_B: cute.TiledCopy,
               tiled_mma: cute.TiledMma):
        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()
        thr_mma = tiled_mma.get_slice(tidx)

        # ----- TODO (A): slice global tensors into per-CTA tiles -----
        # Use cute.local_tile.
        #   gA  shape:  (BLK_M, BLK_K, k)   proj=(1,    None, 1)
        #   gB  shape:  (BLK_N, BLK_K, k)   proj=(None, 1,    1)
        #   gC  shape:  (BLK_M, BLK_N)      proj=(1,    1,    None)
        #
        # coord = (bidx, bidy, None)
        #
        # gA = cute.local_tile(mA, tiler=self.cta_tiler, coord=coord, proj=(1, None, 1))
        # gB = cute.local_tile(mB, tiler=self.cta_tiler, coord=coord, proj=(None, 1, 1))
        # gC = cute.local_tile(mC, tiler=self.cta_tiler, coord=coord, proj=(1, 1, None))
        raise NotImplementedError("Day 10 (A): slice gmem tensors with local_tile")

        # ----- TODO (B): smem allocation + per-thread partitioning -----
        # smem = cutlass.utils.SmemAllocator()
        # sA = smem.allocate_tensor(mA.element_type, sA_layout, 16)
        # sB = smem.allocate_tensor(mB.element_type, sB_layout, 16)
        #
        # thr_copy_A = tiled_copy_A.get_slice(tidx)
        # thr_copy_B = tiled_copy_B.get_slice(tidx)
        # tAgA = thr_copy_A.partition_S(gA)
        # tAsA = thr_copy_A.partition_D(sA)
        # tBgB = thr_copy_B.partition_S(gB)
        # tBsB = thr_copy_B.partition_D(sB)
        #
        # tCsA = thr_mma.partition_A(sA)
        # tCsB = thr_mma.partition_B(sB)
        # tCgC = thr_mma.partition_C(gC)
        # tCrA = tiled_mma.make_fragment_A(tCsA)
        # tCrB = tiled_mma.make_fragment_B(tCsB)
        # tCrC = tiled_mma.make_fragment_C(tCgC)
        # tCrC.fill(0.0)

        # ----- TODO (C): mainloop (synchronous cp.async per K-tile) -----
        # k_tile_count = cute.size(tAgA, mode=[3])
        # k_block_max  = cute.size(tCrA, mode=[2])
        # for k_tile in range(k_tile_count):     # DSL traces this as dynamic
        #     cute.copy(tiled_copy_A,
        #               tAgA[None, None, None, k_tile],
        #               tAsA[None, None, None])
        #     cute.copy(tiled_copy_B,
        #               tBgB[None, None, None, k_tile],
        #               tBsB[None, None, None])
        #     cute.arch.cp_async_commit_group()
        #     cute.arch.cp_async_wait_group(0)
        #     cute.arch.barrier()
        #
        #     for k_block in cutlass.range_constexpr(k_block_max):
        #         cute.autovec_copy(tCsA[None, None, k_block], tCrA[None, None, k_block])
        #         cute.autovec_copy(tCsB[None, None, k_block], tCrB[None, None, k_block])
        #         cute.gemm(tiled_mma, tCrC,
        #                   tCrA[None, None, k_block],
        #                   tCrB[None, None, k_block],
        #                   tCrC)
        #     cute.arch.barrier()
        #
        # # Epilogue: copy accumulator to gmem (unvectorized).
        # atom_store = cute.make_copy_atom(cute.nvgpu.CopyUniversalOp(), mC.element_type)
        # cute.copy(atom_store, tCrC, tCgC)


def run(M: int, N: int, K: int) -> None:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Day 10 needs a CUDA device.")
    cutlass.cuda.initialize_cuda_context()

    a = torch.empty(K, M, dtype=torch.float32).normal_().permute(1, 0).cuda()
    b = torch.empty(K, N, dtype=torch.float32).normal_().permute(1, 0).cuda()
    c = torch.empty(N, M, dtype=torch.float32).zero_().permute(1, 0).cuda()

    a_ = from_dlpack(a, assumed_align=16)
    b_ = from_dlpack(b, assumed_align=16)
    c_ = from_dlpack(c, assumed_align=16)

    print(f"\n=== Day 10 SIMT GEMM, M={M} N={N} K={K} ===")
    SimpleSGemm()(a_, b_, c_)
    torch.cuda.synchronize()

    ref = torch.einsum("mk,nk->mn", a, b)
    torch.testing.assert_close(c.cpu(), ref.cpu(), atol=1e-3, rtol=1e-4)
    print("  OK")

## 11. Correctness before performance

Complete all TODO regions, then run a small problem. The PyTorch
reference uses the same `(M,K) x (N,K)` convention:

```python
torch.einsum("mk,nk->mn", a, b)
```

In [ ]:
run(M=256, N=256, K=64)

## 12. Benchmark and compute throughput

Use CUDA events after warmup. If average duration is in
milliseconds:

```text
GFLOP/s = (2*M*N*K) / (average_ms / 1000) / 1e9
TFLOP/s = GFLOP/s / 1000
```

This SIMT teaching kernel is not expected to approach H100 tensor
core peak. Its value is a transparent baseline whose movement,
synchronization, and ownership can be inspected.

In [ ]:
# The reference script includes warmup, CUDA-event timing, and TFLOP/s.
# Run it after your notebook implementation is correct.
%run solution.py --M 1024 --N 1024 --K 256 --warmup 5 --iterations 50

## 13. Optimization path

The official tour applies two major transformations after its first
correct GEMM:

1. **Software pipeline A/B:** prefetch future K tiles so memory
   latency overlaps current MMA. The stage count and protocol come
   from Day 07.
2. **Vectorize the epilogue:** expose alignment and divisibility so
   CuTe can issue wide global stores.

![Software pipeline](../notebooks/images/software_pipelining_ab_stages_minus_2.svg)

A practical progression for this H100 baseline is:

```text
add edge predication
  -> add two or three A/B stages
  -> overlap copy and compute
  -> vectorize output
  -> replace SIMT MMA with an H100 tensor-core atom
```

Make one transformation at a time, preserve the correctness check,
and use Day 08's benchmark discipline to evaluate it.

## What you should know afterward

- GEMM performance comes from tiling and reuse, not only the FMA.
- `local_tile` selects a CTA problem; partitions assign its work.
- Copy and MMA layouts define ownership before instructions execute.
- Shared memory enables reuse but requires correct producer/consumer ordering.
- Register fragments hold per-thread operands and accumulators.
- A correct single-stage kernel exposes where pipelining can overlap work.
- Hardware-specific SOL kernels preserve these roles while replacing
  mechanisms with TMA, tensor cores, specialized pipelines, and
  vectorized epilogues.